# 01 — Giải thích & Kiểm chứng File Label TuSimple + Pipeline Tiền xử lý ảnh Làn đường CPV301

**Mục tiêu của Notebook này:**
1. **Phần 0 — Giải thích & Kiểm chứng trực quan File Label (Ưu tiên số 1):**
   - Hiểu rõ cấu trúc **Nhãn gốc của bộ dữ liệu TuSimple** (`data/raw/TUSimple/train_set/label_data_*.json` và `test_label.json`): ý nghĩa của `raw_file`, `h_samples` (48 mốc Y từ `240` đến `710`), `lanes` (`-2` vs `x >= 0`), và lý do trong 20 ảnh của mỗi clip (`1.jpg`..`20.jpg`) chỉ có ảnh cuối cùng (`20.jpg`) mang nhãn Ground-Truth (`3.626` ảnh Train + `2.782` ảnh Test = `6.408` ảnh có nhãn trên tổng số `134.571` file `.jpg`).
   - Hiểu rõ quy trình chuyển đổi sang **Nhãn đa giác chuẩn YOLO11-seg** (`data/processing/{train,valid}/labels/*.txt`): ghép điểm `(x, y)` $\rightarrow$ vẽ dải làn dày `12px` $\rightarrow$ trích xuất contour đa giác $\rightarrow$ chuẩn hóa tọa độ về `[0, 1]`.
   - **Hiển thị trực quan song song (Side-by-Side)** giữa ảnh gốc + điểm `(x, y)` từ JSON và ảnh CLAHE + đa giác (`polygon`) từ file `.txt` để kiểm chứng nhãn chính xác 100%.
2. **Phần 1 — Kiểm thử & Trực quan hóa 6 bước Tiền xử lý nâng cao:**
   - **Bước 1:** Cân bằng sáng cục bộ **CLAHE** trên kênh L (không gian màu LAB) + khử nhiễu **Gaussian Blur**.
   - **Bước 2:** Kết hợp phát hiện biên **Canny** và **HLS Adaptive Color Mask + Top-Hat** (lọc vạch kẻ đường trắng & vàng kể cả dưới bóng râm / trời âm u).
   - **Bước 3:** Cắt vùng quan tâm **ROI đa giác (Polygon ROI)** nhìn xa tới chân trời để bắt góc cua sớm.
   - **Bước 4:** Lọc nhiễu đốm nhỏ (**Connected Components Blob Filter**) + **Morphological Closing**.
   - **Bước 5:** Phát hiện đoạn thẳng **Probabilistic Hough Transform**, lọc độ dốc, nội suy đường cong **B-Spline** mượt.
   - **Bước 6:** **Phân loại làn đường chuẩn UK (Left-Hand Traffic)**: xác định biên trái lề đường (`uk_left_edge`), vạch phân cách giữa đường (`uk_center_divider`), và đường dẫn tâm làn trái (`uk_left_lane`).
3. **Phần 2 — Xử lý hàng loạt (Batch Processing) trên toàn bộ Dataset TuSimple có nhãn:**
   - Quét chính xác theo danh sách `raw_file` trong các file JSON (`max_yolo_clips=None`, `max_highlight_images=None`), **tuyệt đối không quét mù 134.571 file `.jpg`**.
   - Áp dụng tiền xử lý **CLAHE (`apply_preprocessing=True`)** khi xuất ảnh sang `data/processing/{train,valid}/images/` để mô hình YOLO11-seg học trên ảnh đã tăng cường tương phản.
   - Lưu bảng thống kê đầy đủ **`preprocessing_metrics.csv`** và biểu đồ **`preprocessing_pipeline_chart.png`** trực tiếp tại thư mục gốc (**Root**).

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tìm thư mục gốc của dự án
ROOT_DIR = Path.cwd().parent if (Path.cwd().parent / "README.md").exists() else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.path import RAW_DATA_DIR, PROCESSING_DATA_DIR
from src.preprocessing.batch_processor import (
    load_or_build_sample_label_verification,
    collect_labeled_road_images,
    preprocess_single_road_image,
    run_batch_preprocessing,
)

print(f"Project Root        : {ROOT_DIR}")
print(f"Raw Data Directory  : {RAW_DATA_DIR}")
print(f"Processing Directory: {PROCESSING_DATA_DIR}")

---
## Phần 0 — Giải thích & Kiểm chứng trực quan File Label (TuSimple JSON vs YOLO11-seg `.txt`)

### 0.1. Cấu trúc File Label gốc của TuSimple (`data/raw/TUSimple/`)
- **Tập Train (`train_set/`):** gồm 3 file `label_data_0313.json` (`2.858` dòng), `label_data_0531.json` (`358` dòng), `label_data_0601.json` (`410` dòng) $\rightarrow$ **Tổng cộng `3.626` ảnh có nhãn**.
- **Tập Test (`test_set/`):** gồm `test_label.json` $\rightarrow$ **`2.782` ảnh có nhãn**.
- **Vì sao mỗi clip có 20 ảnh (`1.jpg` .. `20.jpg`) nhưng chỉ có 1 nhãn?** Mỗi clip là đoạn video dài 1 giây (20 frame), chỉ khung hình cuối cùng (`20.jpg`) được gán nhãn Ground-Truth (`3.626 + 2.782 = 6.408` ảnh có nhãn). 19 ảnh trước đó chỉ phục vụ các thuật toán chuỗi thời gian.
- **Ý nghĩa các trường trong 1 dòng JSON:**
  - `raw_file`: Đường dẫn tương đối tới ảnh `20.jpg` (kích thước chuẩn `1280 x 720`).
  - `h_samples`: Danh sách cố định **48 mốc tọa độ trục dọc `Y`** từ `y = 240` đến `y = 710` (mỗi mốc cách nhau `10px`).
  - `lanes`: Danh sách các làn đường trong ảnh. Mỗi làn có đúng 48 giá trị ứng 1-1 với `h_samples`: giá trị **`-2`** nghĩa là tại độ cao `Y` đó không có vạch kẻ; giá trị **`>= 0`** là tọa độ ngang `X` (pixel) của vạch kẻ đường.

### 0.2. Cấu trúc File Label chuẩn YOLO11-seg (`data/processing/{train,valid}/labels/*.txt`)
- Do YOLO11-seg học phân đoạn theo vùng đa giác có diện tích (Instance Segmentation), hàm `lane_to_polygons()` trong `src/tusimple_to_yolo.py` thực hiện:
  1. Ghép các cặp điểm `(x, y)` hợp lệ (`x >= 0`) của từng làn và vẽ dải dày `thickness = 12px`.
  2. Dùng `cv2.findContours` + `cv2.approxPolyDP` trích xuất các đỉnh đa giác bao quanh dải làn.
  3. Chuẩn hóa tọa độ chia cho `(width=1280, height=720)` về đoạn `[0, 1]` và lưu theo định dạng:
     `<class_id> <x1> <y1> <x2> <y2> ... <xn> <yn>` (với `class_id = 0` ứng với lớp `lane`).

In [ ]:
# Kiểm chứng và trực quan hóa song song: (1) Nhãn gốc JSON TuSimple vs (2) Nhãn đa giác YOLO11-seg (.txt)
verif = load_or_build_sample_label_verification(
    raw_dir=RAW_DATA_DIR,
    processing_dir=PROCESSING_DATA_DIR,
    lane_thickness=12,
)
rec = verif["tusimple_record"]

print("=== 1. KIỂM TRA BẢN GHI NHÃN GỐC TUSIMPLE (JSON) ===")
print(f"- Ảnh đang kiểm tra      : {verif['image_path']} (subset: {verif['subset']})")
print(f"- raw_file               : {rec['raw_file']}")
print(f"- Số mốc cao độ h_samples: {len(rec['h_samples'])} mốc (từ y={rec['h_samples'][0]} đến y={rec['h_samples'][-1]})")
print(f"- Số làn đường trong JSON: {len(rec['lanes'])} làn | Tổng số điểm (x >= 0): {verif['total_gt_points']} điểm")
if rec["lanes"]:
    sample_lane = rec["lanes"][0]
    valid_pairs = [(x, y) for x, y in zip(sample_lane, rec["h_samples"]) if x >= 0][:5]
    print(f"- 5 điểm hợp lệ đầu tiên của làn #1 (X, Y): {valid_pairs}")

print("\n=== 2. KIỂM TRA NHÃN ĐA GIÁC SAU KHI CHUYỂN SANG YOLO11-SEG (.TXT) ===")
print(f"- Số dòng đa giác (.txt) : {len(verif['yolo_lines'])} dòng (mỗi dòng = 1 làn đường class_id=0)")
for idx, line in enumerate(verif["yolo_lines"]):
    tokens = line.split()
    n_vertices = (len(tokens) - 1) // 2
    preview = " ".join(tokens[:9]) + " ..."
    print(f"  Làn #{idx + 1} ({n_vertices:2d} đỉnh): {preview}")

# Hiển thị trực quan song song (Side-by-Side)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].imshow(cv2.cvtColor(verif["json_gt_vis"], cv2.COLOR_BGR2RGB))
axes[0].set_title(
    f"1. Ảnh gốc + Điểm Ground-Truth từ JSON TuSimple\n({len(verif['gt_point_lanes'])} làn, {verif['total_gt_points']} điểm (x, y) ứng với h_samples)",
    fontsize=12,
    fontweight="bold",
)
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(verif["yolo_poly_vis"], cv2.COLOR_BGR2RGB))
axes[1].set_title(
    f"2. Ảnh đã tiền xử lý CLAHE + Đa giác YOLO11-seg (.txt)\n({len(verif['decoded_polygons'])} polygon dày 12px chuẩn hóa [0, 1])",
    fontsize=12,
    fontweight="bold",
)
axes[1].axis("off")

plt.suptitle("Kiểm chứng trực quan: Nhãn điểm TuSimple JSON vs Nhãn đa giác YOLO11-seg (.txt)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Phần 1 — Kiểm thử & Trực quan hóa từng bước Tiền xử lý trên ảnh mẫu

In [ ]:
# Thu thập danh sách các bức ảnh CÓ NHÃN (quét từ JSON của TuSimple + ảnh mẫu trong data/raw, không quét mù 134.571 file)
labeled_entries = collect_labeled_road_images(RAW_DATA_DIR, include_test=True)
assert labeled_entries, f"Không tìm thấy ảnh trong {RAW_DATA_DIR}"

sample_idx = 0
image_path, subset_name = labeled_entries[sample_idx]
image = cv2.imread(str(image_path))

print(f"Tổng số ảnh có nhãn tìm thấy: {len(labeled_entries)}")
print(f"Ảnh mẫu đang chọn           : {image_path} (nguồn: {subset_name})")
print(f"Kích thước ảnh              : {image.shape[1]} x {image.shape[0]} px")

In [ ]:
# Chạy pipeline 6 bước trên ảnh mẫu
stages = preprocess_single_road_image(image)

print("=== THÔNG SỐ PHÂN TÍCH LÀN ĐƯỜNG UK TRÊN ẢNH MẪU ===")
print(f"- Số đoạn thẳng Hough phát hiện : {stages['num_hough_segments']}")
print(f"- Số làn đường B-Spline hội tụ  : {stages['num_detected_lanes']}")
print(f"- Chế độ bám làn UK             : {stages['uk_lane_type']}")
print(f"- Tỷ lệ pixel biên trong ROI    : {stages['edge_validate_ratio']:.4f}")
print(f"- Tỷ lệ pixel vạch trắng/vàng   : {stages['white_mask_ratio']:.4f}")

# Hiển thị trực quan 6 bước xử lý
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
panels = [
    (cv2.cvtColor(stages["original"], cv2.COLOR_BGR2RGB), "1. Ảnh gốc (Original BGR)", None),
    (stages["enhanced_gray"], "2. Cân bằng sáng CLAHE + Khử nhiễu", "gray"),
    (stages["roi_edges"], "3. Biên Canny trong vùng ROI đa giác", "gray"),
    (stages["clean_edges"], "4. Lọc nhiễu Blob + Làm dày hình thái học", "gray"),
    (stages["color_mask"], "5. Mặt nạ màu vạch đường HLS + Top-Hat", "gray"),
    (cv2.cvtColor(stages["highlighted"], cv2.COLOR_BGR2RGB), f"6. Kết quả B-Spline & Làn UK ({stages['uk_lane_type']})", None),
]

for ax, (panel_img, title, cmap) in zip(axes.flat, panels):
    ax.imshow(panel_img, cmap=cmap)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("Pipeline Tiền xử lý Làn đường CPV301 (Chuẩn Giao thông Vương Quốc Anh - LHT)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Phần 2 — Xử lý hàng loạt (Batch Processing) trên toàn bộ Dataset TuSimple có nhãn

Cell dưới đây thực hiện:
1. Quét **toàn bộ các frame có nhãn Ground-Truth** trong `data/raw/TUSimple` theo danh sách `raw_file` từ các file JSON (`max_highlight_images=None`, không giới hạn cứng 60 ảnh và không quét mù 134.571 ảnh) kèm thanh tiến trình `tqdm`.
2. Chuyển đổi **toàn bộ dữ liệu TuSimple (`max_yolo_clips=None`)** kèm bước **tiền xử lý tăng cường tương phản CLAHE (`apply_preprocessing=True`)** sang định dạng **YOLO11-seg** (`data/processing/train/` và `data/processing/valid/`) để phục vụ huấn luyện tại `02_train.ipynb`.
3. Xuất bảng số liệu đầy đủ **`preprocessing_metrics.csv`** và biểu đồ **`preprocessing_pipeline_chart.png`** ra thư mục gốc (`ROOT_DIR`).

In [ ]:
summary = run_batch_preprocessing(
    raw_dir=RAW_DATA_DIR,
    processing_dir=PROCESSING_DATA_DIR,
    root_dir=ROOT_DIR,
    max_highlight_images=None,   # None = chạy trên toàn bộ các ảnh có nhãn của TuSimple
    max_yolo_clips=None,         # None = chuyển đổi toàn bộ 3.626 clip của train_set
    val_ratio=0.2,
    apply_preprocessing=True,    # Áp dụng CLAHE lên toàn bộ ảnh xuất sang data/processing/{train,valid}/images/
)

print("=== KẾT QUẢ XUẤT DỮ LIỆU TIỀN XỬ LÝ ===")
for k, v in summary.items():
    print(f"  {k:28s}: {v}")

In [ ]:
# Đọc và kiểm tra bảng số liệu preprocessing_metrics.csv tại Root
csv_path = ROOT_DIR / "preprocessing_metrics.csv"
df_metrics = pd.read_csv(csv_path)
print(f"Đã lưu CSV tại Root: {csv_path} ({len(df_metrics)} dòng)")
display(df_metrics.head(10))